# GSD guidance ablations and v1 Background completion

Run the checkout cell, then install the PointNet++ CUDA extension before retrying the two failed GSD ablations. The final cell runs only the missing Background corruption for the existing v1 spectral-only M100 seed-0 archive.


In [ ]:
%%bash
set -euo pipefail
cd /content/3DD-TTA
git checkout gsd-smooth-spectrum
git pull --ff-only origin gsd-smooth-spectrum


## 1 - Install the CUDA 12.1 compiler and PointNet++ extension

The previous build selected system CUDA 13.0 while this environment has PyTorch cu121. This cell installs the matching CUDA 12.1 toolkit into 3dd_tta_env, builds PointNet++ for the active GPU, and checks an actual FPS kernel call. Do not rerun requirements.txt.

In [ ]:
%%bash
set -euo pipefail
cd /content/3DD-TTA
conda install -y -n 3dd_tta_env -c nvidia/label/cuda-12.1.1 cuda-toolkit=12.1.1
CUDA_PREFIX="$(conda run --no-capture-output -n 3dd_tta_env python -c 'import sys; print(sys.prefix)')"
test -x "$CUDA_PREFIX/bin/nvcc"
"$CUDA_PREFIX/bin/nvcc" --version
TORCH_CUDA_ARCH_LIST="$(conda run --no-capture-output -n 3dd_tta_env python -c 'import torch; assert torch.cuda.is_available(); print(".".join(map(str, torch.cuda.get_device_capability())))')"
echo "Building PointNet++ for compute capability $TORCH_CUDA_ARCH_LIST"
cd Pointnet2_PyTorch/pointnet2_ops_lib
TORCH_CUDA_ARCH_LIST="$TORCH_CUDA_ARCH_LIST" CUDA_HOME="$CUDA_PREFIX" conda run --no-capture-output -n 3dd_tta_env python setup.py install --force
cd /content/3DD-TTA
conda run --no-capture-output -n 3dd_tta_env python -c 'import torch; import pointnet2_ops._ext; from pointnet2_ops import pointnet2_utils; x=torch.randn(1,16,3,device="cuda").contiguous(); i=pointnet2_utils.furthest_point_sample(x,8); assert tuple(i.shape)==(1,8); print("PointNet++ CUDA import and FPS kernel OK", i.shape)'


## 2 - Guidance-disabled diffusion (3 full runs)

In [ ]:
%%bash
set -euo pipefail
cd /content/3DD-TTA
CALIBRATION="./result/modelnet40_c/gsd_latent_spectral_smooth_v2/20260928-113047_gsd-cal-diagnose-reference-seed0-n64/config.json"
conda run --no-capture-output -n 3dd_tta_env python \
  scripts/run_gsd_unguided.py \
  --calibration "$CALIBRATION" --result-root ./result


## 3 - Smooth spectral-only (3 full runs)

In [ ]:
%%bash
set -euo pipefail
cd /content/3DD-TTA
CALIBRATION="./result/modelnet40_c/gsd_latent_spectral_smooth_v2/20260928-113047_gsd-cal-diagnose-reference-seed0-n64/config.json"
conda run --no-capture-output -n 3dd_tta_env python \
  scripts/run_gsd_smooth_only.py \
  --calibration "$CALIBRATION" --result-root ./result


## 4 - GSD v1 spectral-only M100: missing Background file

Uploads the audited 14-corruption seed-0 ZIP if missing, evaluates only the full 2,468-example Background file, derives the 15-way equal-corruption macro, and appends a labeled entry to knowledge/findings_log.md. The result is a 14+1 cross-run composite.


In [ ]:
from pathlib import Path
import hashlib
from google.colab import files
repo = Path("/content/3DD-TTA").resolve()
target = repo / "result/modelnet40_c/gsd_latent_spectral_v1/20260926-141029_gsd-v1-ablation14-on-seed0-spectral-only.zip"
expected_sha = "a083cf795dc7e2b951fa80f47ea32e939cbeb8f9e41c027449b02431ee6b804a"
if target.exists():
    actual = hashlib.sha256(target.read_bytes()).hexdigest()
    assert actual == expected_sha, "Existing 14-corruption ZIP hash mismatch"
else:
    uploaded = files.upload()
    name = "20260926-141029_gsd-v1-ablation14-on-seed0-spectral-only.zip"
    data = uploaded[name]
    assert hashlib.sha256(data).hexdigest() == expected_sha, "Wrong 14-corruption ZIP"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(data)
print("Audited 14-corruption archive is ready:", target)


In [ ]:
%%bash
set -euo pipefail
cd /content/3DD-TTA
conda run --no-capture-output -n 3dd_tta_env python \
  scripts/run_gsd_v1_background_completion.py \
  --existing-14 ./result/modelnet40_c/gsd_latent_spectral_v1/20260926-141029_gsd-v1-ablation14-on-seed0-spectral-only.zip \
  --result-root ./result


## Output files

- Two guidance-ablation summaries and six raw ZIPs under result/modelnet40_c/gsd_latent_spectral_smooth_v2/.
- One Background ZIP plus spectral_only_m100_all15_summary_seed0.json under result/modelnet40_c/gsd_latent_spectral_v1/.
- A validated seed-0 composite entry is appended to knowledge/findings_log.md.
\nCopy raw ZIPs and derived JSON files off Colab before ending the session.
